# 01 — Instrument calibration (prereg A8 §c + §e)

**BLIND-SAFE. Run this FIRST — every later stage depends on its verdict.**

Scores the **shape anchor only** (the §5 gate factor, orthogonal to H1–H4) plus the
raw-pixel reference, so the confirmatory blind holds. No targeted factor is touched.

Answers the two questions that currently block the paper:

1. **Headroom (A8 §c).** Every Shapes3D random-encoder floor sits at 0.908–0.989 at the
   linear rung, so `G` has under 10% of range and A6(a)'s saturation gate would empty
   the headline family. This sweeps probe-train size and a held-out-factor-value
   (extrapolation) split to find a configuration where the floor drops below 0.90.
2. **Capacity axis (A8 §e).** Cross-checks the ladder's Adam linear rung against a
   convex solver on identical features. They currently disagree by 0.079 on the random
   encoder, and `Delta_G` differences the linear rung against the top rung.

**Setup:** Accelerator `GPU T4 x2`, Internet **On**.
**Output:** `results/calibration/calibration_{shapes3d,dsprites}.json` — read `verdict`
and `recommended_config`, then pin that probe-train size in notebook 03.

## 1. Verify the GPU(s)

In [ ]:
!nvidia-smi

## 2. Clone the repo
Onto `/kaggle/working` (persists across restarts within a session).

In [ ]:
import os

REPO_URL = "https://github.com/chinesegorilla99/probe-capacity-invariance.git"
REPO_DIR = "/kaggle/working/probe-capacity-invariance"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

## 3. Install dependencies
Without disturbing Kaggle's preinstalled, CUDA-matched `torch`/`torchvision`.

In [ ]:
!pip install -q -e . --no-deps
!pip install -q h5py

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "| device count:", torch.cuda.device_count())

## 4. Download shapes3d + dsprites + build the image cache
`--build-cache` decompresses once into an uncompressed memmap the loaders mmap. Idempotent.

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.data.shapes3d --download --build-cache
!cd /kaggle/working/probe-capacity-invariance && python -m src.data.dsprites --download --build-cache

In [ ]:
# Restore prior probe/calibration OUTPUTS (not checkpoints) so a timed-out
# session continues instead of recomputing.
import shutil
from pathlib import Path

REPO = Path("/kaggle/working/probe-capacity-invariance"); INPUT = Path("/kaggle/input")
restored = 0
for src in list(INPUT.glob("*/results")) + list(INPUT.glob("*/probe-capacity-invariance/results")):
    for f in src.rglob("*"):
        if f.is_file() and f.suffix in (".npz", ".json", ".jsonl"):
            dst = REPO / "results" / f.relative_to(src)
            if not dst.exists():
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(f, dst); restored += 1
print(f"restored {restored} prior result files")

## 7. Shapes3D calibration
Extraction dominates; probe fits at n=2000 are ~20x cheaper than at n=40000.

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.probes.instrument_calibration \
    --dataset shapes3d --probe-train-sizes 2000 5000 10000 40000 \
    --random-seed 0 1 2 --device cuda --num-workers 2 --out-root results/calibration

## 8. dSprites calibration (the position arm's dataset)

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.probes.instrument_calibration \
    --dataset dsprites --probe-train-sizes 2000 5000 10000 40000 \
    --random-seed 0 1 2 --device cuda --num-workers 2 --out-root results/calibration

## 9. Read the verdict

In [ ]:
import json
from pathlib import Path
for ds in ("shapes3d", "dsprites"):
    p = Path("/kaggle/working/probe-capacity-invariance") / f"results/calibration/calibration_{ds}.json"
    if not p.exists():
        print(f"{ds}: not run"); continue
    r = json.loads(p.read_text())
    print(f"\n=== {ds} ===\nVERDICT: {r['verdict']}")
    hdr = f"{'regime':16s}{'n':>7s}{'role':>9s}{'top floor':>11s}{'headroom':>10s}{'lin gap':>9s}"
    print(hdr)
    for row in r["results"]:
        print(f"{row['regime']:16s}{row['probe_train_size']:>7d}{row['encoder_role']:>9s}"
              f"{row['top_rung_floor']:>11.4f}{row['headroom_at_top']:>10.4f}"
              f"{row['linear_rung_gap']:>+9.4f}"
              + ("  <- SATURATED" if row["saturated_at_top"] else ""))

In [ ]:
# --- persist for the next session --------------------------------------------
# /kaggle/working is the notebook's output. Click "Save Version" when this
# finishes, then Add Input -> this output on the next run to resume.
import shutil
from pathlib import Path
src = Path("/kaggle/working/probe-capacity-invariance/results"); dst = Path("/kaggle/working/results")
shutil.rmtree(dst, ignore_errors=True); shutil.copytree(src, dst)
print(f"persisted {sum(1 for _ in dst.rglob('*') if _.is_file())} files "
      f"-> click 'Save Version' now")